## 生僻字占位图 Markdown 清洗

遍历配置的 Markdown 目录，将：

1. `![生僻字、公式、图标等）](...)` 替换为纯文本 `[生僻字符]`
2. `![无图名](url)` 改为空 alt：`![](url)`
3. **人工检查**：若某行中 Markdown 图片的 alt 内出现脚注式 `[^…]` 或形如 `[4]` 的纯数字方括号，则在控制台打印**源文件相对路径与行号**（便于核对是否误把正文标进 alt）。

清洗结果写入 `knowledgeBase/md_cleaned_rare_char/` 下对应子目录（不覆盖源文件）。**请在仓库根目录运行本 Notebook**（当前工作目录需包含 `knowledgeBase`）。

In [1]:
import re
from pathlib import Path

PROJECT_ROOT = Path.cwd()

INPUT_OUTPUT_PAIRS = [
    (
        PROJECT_ROOT / "knowledgeBase/liangzhu_unzip/OEBPS/Markdown_2chunk",
        PROJECT_ROOT / "knowledgeBase/md_cleaned_rare_char/liangzhu_Markdown_2chunk",
    ),
    (
        PROJECT_ROOT / "knowledgeBase/wangzhu_unzip/OEBPS/Markdown_Output",
        PROJECT_ROOT / "knowledgeBase/md_cleaned_rare_char/wangzhu_Markdown_Output",
    ),
]

RE_RARE_CHAR_IMG = re.compile(r"!\[生僻字、公式、图标等）\]\([^)]+\)")
RE_NO_CAPTION_IMG = re.compile(r"!\[无图名\]\(([^)]+)\)")
RE_ALT_FOOTNOTE = re.compile(r"\[\^")  # 如 [^1]、[^ref_n]
RE_ALT_BRACKET_NUM = re.compile(r"\[\d+\]")  # 如 [4]、[12]


def iter_markdown_image_alts(line: str):
    """从一行中提取 ![alt](url) 的 alt（按首个 ]( 分界，允许 alt 内含 ]）。"""
    pos = 0
    while True:
        i = line.find("![", pos)
        if i < 0:
            break
        j = i + 2
        k = line.find("](", j)
        if k < 0:
            pos = i + 2
            continue
        alt = line[j:k]
        rp = line.find(")", k + 2)
        if rp < 0:
            pos = i + 2
            continue
        yield alt
        pos = rp + 1


def alt_needs_manual_review(alt: str) -> bool:
    if RE_ALT_FOOTNOTE.search(alt):
        return True
    if RE_ALT_BRACKET_NUM.search(alt):
        return True
    return False


def report_suspicious_image_alts(
    md_path: Path, project_root: Path, lines: list[str]
) -> int:
    """逐行扫描 Markdown 图片 alt，命中则打印相对仓库根的路径与行号。返回命中次数。"""
    n = 0
    rel = md_path.resolve().relative_to(project_root.resolve())
    for lineno, line in enumerate(lines, start=1):
        for alt in iter_markdown_image_alts(line):
            if alt_needs_manual_review(alt):
                n += 1
                print(
                    f"[人工检查] {rel} 第{lineno}行: alt 含 [^…] 或 [数字] → {alt!r}"
                )
    return n


def clean_markdown(text: str) -> tuple[str, int, int]:
    n_rare = len(RE_RARE_CHAR_IMG.findall(text))
    text = RE_RARE_CHAR_IMG.sub("[生僻字符]", text)
    n_noname = len(RE_NO_CAPTION_IMG.findall(text))
    text = RE_NO_CAPTION_IMG.sub(r"![](\1)", text)
    return text, n_rare, n_noname


total_rare = total_noname = 0
files_changed = 0
total_alt_flags = 0

for src_root, dst_root in INPUT_OUTPUT_PAIRS:
    if not src_root.is_dir():
        print(f"跳过（源目录不存在）: {src_root}")
        continue
    md_list = sorted(src_root.rglob("*.md"))
    print(f"\n处理: {src_root.relative_to(PROJECT_ROOT)} — 共 {len(md_list)} 个 .md")
    print("  （扫描图片 alt：若含 [^…] 或 [数字] 将打印 [人工检查] 行）")
    for md_path in md_list:
        rel = md_path.relative_to(src_root)
        out_path = dst_root / rel
        out_path.parent.mkdir(parents=True, exist_ok=True)
        raw = md_path.read_text(encoding="utf-8")
        lines = raw.splitlines()
        total_alt_flags += report_suspicious_image_alts(md_path, PROJECT_ROOT, lines)
        cleaned, n_rare, n_noname = clean_markdown(raw)
        out_path.write_text(cleaned, encoding="utf-8")
        total_rare += n_rare
        total_noname += n_noname
        if n_rare or n_noname:
            files_changed += 1
            print(f"  {rel}: 生僻占位={n_rare}, 无图名→空alt={n_noname}")

print("\n" + "=" * 60)
print(
    f"完成。生僻占位替换总计: {total_rare}；无图名空 alt 总计: {total_noname}；"
    f"至少有一处替换的文件数: {files_changed}"
)
print(f"图片 alt 需人工检查（[^…] 或 [数字]）的命中次数: {total_alt_flags}")
print(f"输出根目录: {PROJECT_ROOT / 'knowledgeBase/md_cleaned_rare_char'}")


处理: knowledgeBase\liangzhu_unzip\OEBPS\Markdown_2chunk — 共 34 个 .md
  （扫描图片 alt：若含 [^…] 或 [数字] 将打印 [人工检查] 行）
  Ssc04_0001.md: 生僻占位=36, 无图名→空alt=0
  Ssc05_0001.md: 生僻占位=42, 无图名→空alt=12
  Ssc05_0002.md: 生僻占位=71, 无图名→空alt=20
  Ssc06_0001.md: 生僻占位=11, 无图名→空alt=32
  Ssc07_0001.md: 生僻占位=12, 无图名→空alt=1
  Ssc07_0002.md: 生僻占位=21, 无图名→空alt=51
  Ssc08_0001.md: 生僻占位=56, 无图名→空alt=7
  Ssc08_0002.md: 生僻占位=48, 无图名→空alt=7
  Ssc08_0003.md: 生僻占位=26, 无图名→空alt=10
  Ssc08_0004.md: 生僻占位=20, 无图名→空alt=0
  Ssc08_0005.md: 生僻占位=24, 无图名→空alt=0
  Ssc08_0006.md: 生僻占位=20, 无图名→空alt=1
  Ssc08_0007.md: 生僻占位=7, 无图名→空alt=0
  Ssc08_0008.md: 生僻占位=37, 无图名→空alt=0
  Ssc08_0009.md: 生僻占位=12, 无图名→空alt=5
  Ssc08_0010.md: 生僻占位=25, 无图名→空alt=0
  Ssc10_0001.md: 生僻占位=16, 无图名→空alt=0
  Ssc10_0004.md: 生僻占位=4, 无图名→空alt=0
  Ssc10_0005.md: 生僻占位=2, 无图名→空alt=0
  Ssc10_0006.md: 生僻占位=22, 无图名→空alt=0
  Ssc10_0007.md: 生僻占位=16, 无图名→空alt=0
  Ssc10_0008.md: 生僻占位=4, 无图名→空alt=0
  Ssc10_0009.md: 生僻占位=2, 无图名→空alt=0
  Ssc10_0010.md: 生僻占位=1, 无图名→空alt=0
  S